In [1]:

# Import libraries for database access, data processing,
# statistical hypothesis testing, and interval estimation.
import pyodbc
import pandas as pd
from scipy.stats import chi2_contingency
from math import sqrt

# Connect to our existing SQL Server analytics database.
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=Sindhu;"
    "DATABASE=OlistEcommerceAnalytics;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Aggregate order-level review outcomes in SQL Server.
# Select only the latest review per order to avoid duplication.
query = """
WITH ranked_reviews AS (
    SELECT
        order_id,
        review_score,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY
                review_answer_timestamp DESC,
                review_creation_date DESC,
                review_id DESC
        ) AS rn
    FROM dw.fact_order_reviews
)
SELECT
    o.is_late,
    COUNT(*) AS eligible_orders,
    COUNT(r.review_score) AS reviewed_orders,
    SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END)
        AS low_ratings
FROM dw.fact_orders AS o
LEFT JOIN ranked_reviews AS r
    ON o.order_id = r.order_id AND r.rn = 1
WHERE o.delivery_kpi_eligible = 1
GROUP BY o.is_late
"""

# Retrieve aggregated counts from the SQL database.
with pyodbc.connect(connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    records = [tuple(row) for row in cursor.fetchall()]

# Create a readable analytical summary.
df = pd.DataFrame(
    records,
    columns=[
        "is_late", "eligible_orders",
        "reviewed_orders", "low_ratings"
    ]
)

# Derive non-low-rated orders and review coverage.
df["other_ratings"] = (
    df["reviewed_orders"] - df["low_ratings"]
)

df["review_coverage_pct"] = (
    100 * df["reviewed_orders"] / df["eligible_orders"]
)

# Show the input counts used by the statistical test.
print("DELIVERY SATISFACTION — STATISTICAL INPUT DATA")
display(df)

# Construct the 2x2 contingency table using reviewed orders.
on_time = df.loc[df["is_late"] == 0].iloc[0]
late = df.loc[df["is_late"] == 1].iloc[0]

table = [
    [int(on_time["low_ratings"]), int(on_time["other_ratings"])],
    [int(late["low_ratings"]), int(late["other_ratings"])]
]

# Test the independence hypothesis.
chi2, p_value, dof, expected = chi2_contingency(
    table, correction=False
)

# Measure the practical size of the observed association.
p_on_time = on_time["low_ratings"] / on_time["reviewed_orders"]
p_late = late["low_ratings"] / late["reviewed_orders"]

risk_difference = p_late - p_on_time
risk_ratio = p_late / p_on_time

# Calculate an approximate 95% confidence interval
# for the difference between two independent proportions.
standard_error = sqrt(
    p_late * (1 - p_late) / late["reviewed_orders"]
    + p_on_time * (1 - p_on_time) / on_time["reviewed_orders"]
)

ci_lower = risk_difference - 1.96 * standard_error
ci_upper = risk_difference + 1.96 * standard_error

# Display the statistical results.
print("\nCHI-SQUARE HYPOTHESIS TEST")
print("Chi-square statistic:", round(chi2, 2))
print("Degrees of freedom:", dof)
print("P-value:", f"{p_value:.3e}")
print("Risk difference (percentage points):",
      round(risk_difference * 100, 2))
print("Relative risk:", round(risk_ratio, 2))
print("95% CI for risk difference:",
      f"[{ci_lower * 100:.2f}, {ci_upper * 100:.2f}] pp")

# Report the statistical decision without claiming causation.
if p_value < 0.05:
    print("Decision: Reject the null hypothesis.")
else:
    print("Decision: Insufficient evidence to reject H0.")


DELIVERY SATISFACTION — STATISTICAL INPUT DATA


,is_late,eligible_orders,reviewed_orders,low_ratings,other_ratings,review_coverage_pct
0,False,89936,89443,8289,81154,99.451832
1,True,6534,6381,3983,2398,97.658402



CHI-SQUARE HYPOTHESIS TEST
Chi-square statistic: 15068.91
Degrees of freedom: 1
P-value: 0.000e+00
Risk difference (percentage points): 53.15
Relative risk: 6.74
95% CI for risk difference: [51.95, 54.36] pp
Decision: Reject the null hypothesis.


In [3]:

# Project: Olist E-Commerce Analytics
# Purpose: Estimate the adjusted association between
# late delivery and low customer ratings.

import pandas as pd
import numpy as np
import pyodbc
import statsmodels.api as sm
import statsmodels.formula.api as smf

from IPython.display import display

# Connect to the same SQL Server warehouse.
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=Sindhu;"
    "DATABASE=OlistEcommerceAnalytics;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Retrieve the previously validated order-level features.
query = """
SELECT
    order_id,
    order_purchase_timestamp,
    customer_state,
    is_late,
    low_rating_flag,
    item_count,
    seller_count,
    order_item_value_brl,
    order_freight_brl,
    promised_delivery_days
FROM analytics.vw_delivery_satisfaction_features;
"""

# Load query results into a pandas DataFrame.
with pyodbc.connect(connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute(query)

    columns = [col[0] for col in cursor.description]
    rows = [tuple(row) for row in cursor.fetchall()]

df = pd.DataFrame(rows, columns=columns)

# Verify that we retrieved the correct dataset.
print("Rows retrieved:", len(df))
print("Unique orders:", df["order_id"].nunique())

assert len(df) == 95824
assert df["order_id"].is_unique

# Convert SQL numeric types into Python numeric values.
numeric_columns = [
    "is_late",
    "low_rating_flag",
    "item_count",
    "seller_count",
    "order_item_value_brl",
    "order_freight_brl",
    "promised_delivery_days"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column], errors="coerce"
    )

# Extract purchase month to account for seasonality.
df["purchase_month"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
).dt.month

# Identify rows with incomplete or invalid modeling inputs.
required_columns = (
    numeric_columns + ["customer_state", "purchase_month"]
)

valid = (
    df[required_columns].notna().all(axis=1)
    & (df["order_item_value_brl"] >= 0)
    & (df["order_freight_brl"] >= 0)
    & (df["promised_delivery_days"] >= 0)
    & df["is_late"].isin([0, 1])
    & df["low_rating_flag"].isin([0, 1])
)

# Retain valid observations and disclose exclusions.
model_df = df.loc[valid].copy()

print("Rows excluded:", len(df) - len(model_df))
print("Rows used for regression:", len(model_df))

# Ensure the outcome and exposure are binary integers.
model_df["is_late"] = model_df["is_late"].astype(int)
model_df["low_rating_flag"] = (
    model_df["low_rating_flag"].astype(int)
)

# Fit a binomial logistic regression.
# Log transforms reduce the influence of very large
# order and freight values.
formula = """
low_rating_flag ~
    is_late
    + np.log1p(order_item_value_brl)
    + np.log1p(order_freight_brl)
    + item_count
    + seller_count
    + promised_delivery_days
    + C(customer_state)
    + C(purchase_month)
"""

# HC3 provides heteroskedasticity-robust standard errors.
model = smf.glm(
    formula=formula,
    data=model_df,
    family=sm.families.Binomial()
).fit(cov_type="HC3", maxiter=100)

# Extract the adjusted late-delivery coefficient.
coefficient = model.params["is_late"]
standard_error = model.bse["is_late"]

# Convert log-odds into an adjusted odds ratio.
adjusted_or = np.exp(coefficient)

# Construct an approximate 95% confidence interval.
ci_lower = np.exp(coefficient - 1.96 * standard_error)
ci_upper = np.exp(coefficient + 1.96 * standard_error)

# Summarize the main explanatory result.
results = pd.DataFrame({
    "Metric": [
        "Adjusted Odds Ratio",
        "95% CI Lower",
        "95% CI Upper",
        "P-value",
        "Observations"
    ],
    "Value": [
        adjusted_or,
        ci_lower,
        ci_upper,
        model.pvalues["is_late"],
        model.nobs
    ]
})

print("\nADJUSTED LOGISTIC REGRESSION RESULTS")
display(results.round(4))


Rows retrieved: 95824
Unique orders: 95824
Rows excluded: 0
Rows used for regression: 95824

ADJUSTED LOGISTIC REGRESSION RESULTS


,Metric,Value
0,Adjusted Odds Ratio,17.2294
1,95% CI Lower,16.2394
2,95% CI Upper,18.2797
3,P-value,0.0000
4,Observations,95824.0000


In [4]:

# Purpose: Validate logistic regression convergence and
# translate adjusted odds into model-based probabilities.
# Verify that the logistic regression algorithm converged.
print("Model converged:", model.converged)
print("Observations used:", int(model.nobs))

# Calculate the unadjusted odds ratio from observed outcomes.
summary = (
    model_df.groupby("is_late")["low_rating_flag"]
    .agg(["sum", "count"])
)

# Separate low-rated and other-rated orders for each group.
on_time_low = summary.loc[0, "sum"]
on_time_other = summary.loc[0, "count"] - on_time_low

late_low = summary.loc[1, "sum"]
late_other = summary.loc[1, "count"] - late_low

# Calculate the crude odds ratio without adjusting
# for any additional explanatory variables.
crude_odds_ratio = (
    (late_low / late_other)
    / (on_time_low / on_time_other)
)

# Extract the adjusted odds ratio from our regression.
adjusted_odds_ratio = np.exp(model.params["is_late"])

print("\nODDS RATIO COMPARISON")
print("Unadjusted odds ratio:",
      round(crude_odds_ratio, 2))
print("Adjusted odds ratio:",
      round(adjusted_odds_ratio, 2))

# Create two copies with identical observed control variables.
# Change only the delivery status to compare model predictions.
on_time_scenario = model_df.copy()
late_scenario = model_df.copy()

on_time_scenario["is_late"] = 0
late_scenario["is_late"] = 1

# Generate predicted low-rating probabilities under
# each hypothetical delivery-status setting.
prob_on_time = model.predict(on_time_scenario).mean()
prob_late = model.predict(late_scenario).mean()

# Summarize model-based adjusted probabilities.
adjusted_results = pd.DataFrame({
    "Delivery Status": ["On-Time", "Late"],
    "Adjusted Low-Rating Probability (%)": [
        round(prob_on_time * 100, 2),
        round(prob_late * 100, 2)
    ]
})

print("\nMODEL-BASED ADJUSTED PROBABILITIES")
display(adjusted_results)

# Calculate the model-based probability difference.
print(
    "Adjusted probability difference (pp):",
    round((prob_late - prob_on_time) * 100, 2)
)


Model converged: True
Observations used: 95824

ODDS RATIO COMPARISON
Unadjusted odds ratio: 16.26
Adjusted odds ratio: 17.23

MODEL-BASED ADJUSTED PROBABILITIES


,Delivery Status,Adjusted Low-Rating Probability (%)
0,On-Time,9.27
1,Late,61.25


Adjusted probability difference (pp): 51.97


In [5]:

# Display both model-adjusted low-rating probabilities
# from our previous logistic regression analysis.

print("MODEL-ADJUSTED LOW-RATING PROBABILITIES")
print("-" * 45)

# Show the average predicted probability under each scenario.
print(f"On-Time Delivery: {prob_on_time * 100:.2f}%")
print(f"Late Delivery:    {prob_late * 100:.2f}%")

# Calculate their difference in percentage points.
adjusted_difference = (
    prob_late - prob_on_time
) * 100

print(f"Difference:       {adjusted_difference:.2f} pp")


MODEL-ADJUSTED LOW-RATING PROBABILITIES
---------------------------------------------
On-Time Delivery: 9.27%
Late Delivery:    61.25%
Difference:       51.97 pp
